In [0]:
from pyspark.sql.functions import col, to_date, when, trim, upper, current_timestamp, lower, collect_list

In [0]:
catalog = "clinical_trials"
bronze_schema = "bronze"
silver_schema = "silver"

print("Configuration loaded.")
print(f"Source: {catalog}.{bronze_schema}")
print(f"Target: {catalog}.{silver_schema}")

Configuration loaded.
Source: clinical_trials.bronze
Target: clinical_trials.silver


In [0]:
bronze_studies = spark.table(f"{catalog}.{bronze_schema}.bronze_studies")

silver_studies = (bronze_studies
    # Cast dates
    .withColumn("start_date", to_date(col("start_date")))
    .withColumn("completion_date", to_date(col("completion_date")))
    .withColumn("primary_completion_date", to_date(col("primary_completion_date")))
    .withColumn("study_first_submitted_date", to_date(col("study_first_submitted_date")))
    .withColumn("results_first_submitted_date", to_date(col("results_first_submitted_date")))
    # Cast numerics
    .withColumn("enrollment", col("enrollment").cast("int"))
    .withColumn("number_of_arms", col("number_of_arms").cast("int"))
    # Standardize categoricals
    .withColumn("overall_status", trim(upper(col("overall_status"))))
    .withColumn("study_type", trim(upper(col("study_type"))))
    .withColumn("phase", trim(upper(col("phase"))))
    .withColumn("enrollment_type", trim(upper(col("enrollment_type"))))
    .withColumn("source_class", trim(upper(col("source_class"))))
    # Cast booleans
    .withColumn("has_dmc", when(col("has_dmc") == "t", True)
                           .when(col("has_dmc") == "f", False)
                           .otherwise(None))
    .withColumn("is_fda_regulated_drug", when(col("is_fda_regulated_drug") == "t", True)
                                         .when(col("is_fda_regulated_drug") == "f", False)
                                         .otherwise(None))
    .withColumn("is_fda_regulated_device", when(col("is_fda_regulated_device") == "t", True)
                                           .when(col("is_fda_regulated_device") == "f", False)
                                           .otherwise(None))
    # Remove rows with null nct_id
    .filter(col("nct_id").isNotNull())
    # Select only columns we need
    .select(
        "nct_id", "overall_status", "study_type", "phase",
        "start_date", "completion_date", "primary_completion_date",
        "study_first_submitted_date", "results_first_submitted_date",
        "enrollment", "enrollment_type", "number_of_arms",
        "source_class", "has_dmc", "is_fda_regulated_drug",
        "is_fda_regulated_device", "brief_title", "source",
        current_timestamp().alias("_transformed_at")
    )
)

# Write to silver
(silver_studies.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{silver_schema}.silver_studies"))

count = spark.table(f"{catalog}.{silver_schema}.silver_studies").count()
print(f"✓ silver_studies: {count:,} rows")

✓ silver_studies: 587,788 rows


In [0]:
bronze_calc = spark.table(f"{catalog}.{bronze_schema}.bronze_calculated_values")

silver_calc = (bronze_calc
    # Cast numerics
    .withColumn("actual_duration", col("actual_duration").cast("int"))
    .withColumn("number_of_facilities", col("number_of_facilities").cast("int"))
    .withColumn("minimum_age_num", col("minimum_age_num").cast("int"))
    .withColumn("maximum_age_num", col("maximum_age_num").cast("int"))
    .withColumn("number_of_primary_outcomes_to_measure", 
                col("number_of_primary_outcomes_to_measure").cast("int"))
    .withColumn("number_of_secondary_outcomes_to_measure", 
                col("number_of_secondary_outcomes_to_measure").cast("int"))
    .withColumn("registered_in_calendar_year", 
                col("registered_in_calendar_year").cast("int"))
    # Cast booleans
    .withColumn("were_results_reported", 
                when(col("were_results_reported") == "t", True)
                .when(col("were_results_reported") == "f", False)
                .otherwise(None))
    .withColumn("has_us_facility", 
                when(col("has_us_facility") == "t", True)
                .when(col("has_us_facility") == "f", False)
                .otherwise(None))
    .withColumn("has_single_facility", 
                when(col("has_single_facility") == "t", True)
                .when(col("has_single_facility") == "f", False)
                .otherwise(None))
    # Handle age nulls with semantic fill
    .withColumn("minimum_age_num", 
                when(col("minimum_age_num").isNull(), 0)
                .otherwise(col("minimum_age_num")))
    .withColumn("maximum_age_num", 
                when(col("maximum_age_num").isNull(), 999)
                .otherwise(col("maximum_age_num")))
    # Derive age restriction flag
    .withColumn("has_age_restriction",
                (col("minimum_age_num") > 0) | (col("maximum_age_num") < 999))
    # Remove rows with null nct_id
    .filter(col("nct_id").isNotNull())
    # Select columns
    .select(
        "nct_id", "actual_duration", "number_of_facilities",
        "minimum_age_num", "maximum_age_num", "has_age_restriction",
        "were_results_reported", "has_us_facility", "has_single_facility",
        "registered_in_calendar_year",
        "number_of_primary_outcomes_to_measure",
        "number_of_secondary_outcomes_to_measure",
        current_timestamp().alias("_transformed_at")
    )
)

(silver_calc.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{silver_schema}.silver_calculated_values"))

count = spark.table(f"{catalog}.{silver_schema}.silver_calculated_values").count()
print(f"✓ silver_calculated_values: {count:,} rows")

✓ silver_calculated_values: 587,788 rows


In [0]:
bronze_conditions = spark.table(f"{catalog}.{bronze_schema}.bronze_conditions")

silver_conditions = (bronze_conditions
    .filter(col("nct_id").isNotNull())
    .filter(col("name").isNotNull())
    .withColumn("condition_name", trim(lower(col("name"))))
    .groupBy("nct_id")
    .agg(collect_list("condition_name").alias("conditions"))
    .withColumn("_transformed_at", current_timestamp())
)

(silver_conditions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{silver_schema}.silver_conditions"))

count = spark.table(f"{catalog}.{silver_schema}.silver_conditions").count()
print(f"✓ silver_conditions: {count:,} rows")

✓ silver_conditions: 586,779 rows


In [0]:
bronze_sponsors = spark.table(f"{catalog}.{bronze_schema}.bronze_sponsors")

silver_sponsors = (bronze_sponsors
    .filter(col("nct_id").isNotNull())
    # Keep lead sponsors only
    .filter(col("lead_or_collaborator") == "lead")
    .withColumn("agency_class", trim(upper(col("agency_class"))))
    .select(
        "nct_id",
        "agency_class",
        "name",
        current_timestamp().alias("_transformed_at")
    )
)

(silver_sponsors.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{silver_schema}.silver_sponsors"))

count = spark.table(f"{catalog}.{silver_schema}.silver_sponsors").count()
print(f"✓ silver_sponsors: {count:,} rows")

✓ silver_sponsors: 587,788 rows


In [0]:
print("=== Data Quality Report ===\n")

# silver_studies
s = spark.table(f"{catalog}.{silver_schema}.silver_studies")
print("silver_studies:")
print(f"  Total rows: {s.count():,}")
assert s.filter(col("nct_id").isNull()).count() == 0, "NULL nct_ids found in silver_studies"
print(f"  ✓ No null nct_ids")
print(f"  Null start_date: {s.filter(col('start_date').isNull()).count():,}")
print(f"  Null completion_date: {s.filter(col('completion_date').isNull()).count():,}")
print(f"  Null enrollment: {s.filter(col('enrollment').isNull()).count():,}")

# silver_calculated_values
c = spark.table(f"{catalog}.{silver_schema}.silver_calculated_values")
print("\nsilver_calculated_values:")
print(f"  Total rows: {c.count():,}")
assert c.filter(col("nct_id").isNull()).count() == 0, "NULL nct_ids found in silver_calculated_values"
print(f"  ✓ No null nct_ids")
print(f"  Null actual_duration: {c.filter(col('actual_duration').isNull()).count():,}")
print(f"  Min minimum_age_num: {c.selectExpr('min(minimum_age_num)').collect()[0][0]}")
print(f"  Max maximum_age_num: {c.selectExpr('max(maximum_age_num)').collect()[0][0]}")
print(f"  Trials with age restriction: {c.filter(col('has_age_restriction')).count():,}")

# silver_conditions
cond = spark.table(f"{catalog}.{silver_schema}.silver_conditions")
print("\nsilver_conditions:")
print(f"  Total unique trials with conditions: {cond.count():,}")

# silver_sponsors
sp = spark.table(f"{catalog}.{silver_schema}.silver_sponsors")
print("\nsilver_sponsors:")
print(f"  Total lead sponsor records: {sp.count():,}")
print(f"  Null agency_class: {sp.filter(col('agency_class').isNull()).count():,}")

print("\n=== All checks passed ===")

=== Data Quality Report ===

silver_studies:
  Total rows: 587,788
  ✓ No null nct_ids
  Null start_date: 5,339
  Null completion_date: 16,701
  Null enrollment: 7,110

silver_calculated_values:
  Total rows: 587,788
  ✓ No null nct_ids
  Null actual_duration: 227,543
  Min minimum_age_num: 0
  Max maximum_age_num: 6569
  Trials with age restriction: 557,605

silver_conditions:
  Total unique trials with conditions: 586,779

silver_sponsors:
  Total lead sponsor records: 587,788
  Null agency_class: 975

=== All checks passed ===


In [0]:
silver_calc = spark.table(f"{catalog}.{silver_schema}.silver_calculated_values")

# Check how many rows are affected
outliers = silver_calc.filter(col("maximum_age_num") > 100).count()
print(f"Rows with maximum_age_num > 100: {outliers:,}")

# Show the distribution of outlier values
print("\nOutlier value distribution:")
display(
    silver_calc
    .filter(col("maximum_age_num") > 100)
    .groupBy("maximum_age_num")
    .count()
    .orderBy("maximum_age_num", ascending=False)
    .limit(20)
)

Rows with maximum_age_num > 100: 279,721

Outlier value distribution:


maximum_age_num,count
6569,1
4383,1
2190,1
2189,1
1824,1
1095,1
999,275717
915,1
730,1
578,1


In [0]:
silver_calc_fixed = (silver_calc
    .withColumn("maximum_age_num",
        when(col("maximum_age_num") == 999, 999)  # keep our sentinel value
        .when(col("maximum_age_num") > 100, None)  # null out true outliers
        .otherwise(col("maximum_age_num")))
    # Recalculate has_age_restriction since maximum_age_num changed
    .withColumn("has_age_restriction",
        (col("minimum_age_num") > 0) | 
        ((col("maximum_age_num") < 999) & col("maximum_age_num").isNotNull()))
)

# Verify fix
print("After fix:")
print(f"  Max maximum_age_num (excluding 999): {silver_calc_fixed.filter(col('maximum_age_num') != 999).selectExpr('max(maximum_age_num)').collect()[0][0]}")
print(f"  Null maximum_age_num: {silver_calc_fixed.filter(col('maximum_age_num').isNull()).count():,}")

# Overwrite the silver table
(silver_calc_fixed.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{silver_schema}.silver_calculated_values"))

print("✓ silver_calculated_values updated")

After fix:
  Max maximum_age_num (excluding 999): 100
  Null maximum_age_num: 4,004
✓ silver_calculated_values updated


In [0]:
print("=== Final Silver Layer Summary ===\n")

tables = [
    "silver_studies",
    "silver_calculated_values", 
    "silver_conditions",
    "silver_sponsors"
]

for table in tables:
    df = spark.table(f"{catalog}.{silver_schema}.{table}")
    print(f"{table}:")
    print(f"  Rows: {df.count():,}")
    print(f"  Columns: {len(df.columns)}")
    print()

=== Final Silver Layer Summary ===

silver_studies:
  Rows: 587,788
  Columns: 19

silver_calculated_values:
  Rows: 587,788
  Columns: 13

silver_conditions:
  Rows: 586,779
  Columns: 3

silver_sponsors:
  Rows: 587,788
  Columns: 4

